# 10 Caching and Persistence

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand <code>cache()</code> and <code>persist()</code> (storage levels, laziness, <code>unpersist</code>), when caching helps and when it hurts, and what to use instead on Databricks serverless, where the caching APIs aren't available: materialized Delta tables and the automatic disk cache.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Because every action recomputes its plan (lesson 04), reusing an expensive DataFrame several times can multiply the cost. Caching is the classic answer, and "cache vs persist?" and "when would you not cache?" are frequent interview questions. On Databricks you also need to know the modern alternatives, because serverless compute doesn't support <code>cache()</code>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How caching works</b>
<ul><li><code>df.cache()</code> marks the DataFrame to be stored the <b>next time an action computes it</b>. It is <b>lazy</b></li><li><code>df.persist(level)</code> does the same with a chosen <b>storage level</b>. For DataFrames, <code>cache()</code> = <code>persist(StorageLevel.MEMORY_AND_DISK)</code></li><li>Cached data lives in <b>executor</b> memory (and disk), per partition. It's lost if the executor dies, and Spark recomputes those partitions from the lineage</li><li><code>df.unpersist()</code> frees it. Cached data uses memory that joins and aggregations would otherwise use</li></ul>
</div>

| Storage level | Where | Notes |
|---|---|---|
| `MEMORY_ONLY` | Executor memory | Partitions that don't fit are recomputed when needed |
| `MEMORY_AND_DISK` (default for DataFrames) | Memory, spill to local disk | Safe default |
| `DISK_ONLY` | Local disk | For large data that's expensive to recompute |
| `*_2` variants | Two replicas | Faster recovery, double the space |
| `OFF_HEAP` | Off-heap memory | Needs off-heap configuration |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An ML feature job reads 300 GB, joins 4 tables, and then computes 25 different aggregates from the joined result in a loop. Without caching, the 4-way join ran 25 times. Caching the joined DataFrame (on classic compute) cut runtime from 5 hours to 40 minutes. On serverless, the same fix is to <b>write the joined result to a Delta table</b> once and read it 25 times.
</div>

## Syntax

```python
from pyspark import StorageLevel

df.cache()                                   # MEMORY_AND_DISK, lazy
df.persist(StorageLevel.DISK_ONLY)           # choose a level, lazy
df.count()                                   # an action materializes the cache
df.storageLevel, df.is_cached                # inspect
df.unpersist()                               # free

spark.sql("CACHE TABLE sales")               # SQL (eager by default)
spark.sql("UNCACHE TABLE sales")

# Serverless alternative: materialize to Delta
df.write.mode("overwrite").saveAsTable("workspace.spark_basics.tmp_result")
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an expensive DataFrame (a join plus aggregation) and a helper to check whether caching is supported on this compute.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>caching supported: True</code> on classic compute (and locally), <code>False</code> on serverless.
</div>

In [0]:
import time
from pyspark import StorageLevel
from pyspark.sql import functions as F

sales = spark.range(0, 4_000_000, 1, numPartitions=8).select(
    (F.col("id") % 50_000).alias("customer_id"), (F.col("id") % 997).alias("amount"))
customers = spark.range(50_000).select(F.col("id").alias("customer_id"), (F.col("id") % 25).alias("segment"))

expensive = (
    sales.join(customers.hint("merge"), "customer_id")
    .groupBy("customer_id", "segment").agg(F.sum("amount").alias("total"), F.count("*").alias("orders"))
)

def caching_supported():
    try:
        probe = spark.range(1).cache()
        probe.count()
        probe.unpersist()
        return True
    except Exception:
        return False

CAN_CACHE = caching_supported()
print("caching supported:", CAN_CACHE)

def timed(label, fn):
    start = time.time(); fn(); print(f"{label:<40} {time.time() - start:.2f}s")

## 1. Without caching: every action recomputes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs three different actions on the expensive DataFrame and times each.<br><br>
<b>Why it matters</b><br>This is the baseline. Each action re-runs the join and aggregation from scratch.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three similar, non-trivial timings.
</div>

In [0]:
timed("action 1: count", lambda: expensive.count())
timed("action 2: segments > 10", lambda: expensive.filter("segment > 10").count())
timed("action 3: top customer", lambda: expensive.orderBy(F.desc("total")).first())

## 2. With caching (classic compute)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Caches the DataFrame, materializes it with one action, and reruns the three actions.<br><br>
<b>Why it matters</b><br>The first action pays to compute <b>and</b> store the data. Later actions read from the cache, which is much faster. <code>cache()</code> itself returns instantly because it's lazy.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
On serverless, this cell prints that caching isn't available and skips the demo. The concept is still important for classic clusters and for interviews.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On classic compute: <code>cache()</code> returns instantly, the materializing count is slow, and the next three actions are fast. The storage level shows disk and memory.
</div>

In [0]:
if CAN_CACHE:
    cached = expensive.cache()
    print("storage level:", cached.storageLevel)
    timed("materialize (first action)", lambda: cached.count())
    timed("action 1: count (from cache)", lambda: cached.count())
    timed("action 2: segments > 10 (from cache)", lambda: cached.filter("segment > 10").count())
    timed("action 3: top customer (from cache)", lambda: cached.orderBy(F.desc("total")).first())
    cached.unpersist()
else:
    print("cache() is not supported on this compute (serverless). See section 4 for the alternative.")

## 3. Storage levels and SQL caching

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Persists with <code>DISK_ONLY</code>, inspects the level, unpersists, and shows the SQL <code>CACHE TABLE</code> statement.<br><br>
<b>Why it matters</b><br>Choose the level by trade-off: memory is fastest but limited, and disk survives memory pressure. In SQL, <code>CACHE TABLE</code> is <b>eager</b> by default (it caches immediately), unlike <code>df.cache()</code>. Use <code>CACHE LAZY TABLE</code> for lazy behaviour.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On classic compute: <code>Disk Serialized 1x Replicated</code>, then <code>is_cached</code> false after unpersist, then a cached view count. On serverless: a "not supported" note.
</div>

In [0]:
if CAN_CACHE:
    on_disk = expensive.persist(StorageLevel.DISK_ONLY)
    on_disk.count()
    print("level:", on_disk.storageLevel, "| is_cached:", on_disk.is_cached)
    on_disk.unpersist()
    print("after unpersist, is_cached:", on_disk.is_cached)

    expensive.createOrReplaceTempView("customer_totals")
    spark.sql("CACHE TABLE customer_totals")
    print("cached view rows:", spark.table("customer_totals").count())
    spark.sql("UNCACHE TABLE customer_totals")
else:
    print("persist / CACHE TABLE are not supported on this compute.")

## 4. The serverless way: materialize to Delta

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the expensive result once to a Delta table, then runs the same three actions on the table.<br><br>
<b>Why it matters</b><br>A Delta table is a durable, shareable "cache": it survives restarts and can be read by other notebooks and job tasks. On Databricks it's also read through the automatic <b>disk cache</b> (section 5), so repeated reads are fast. It's the recommended pattern for reused intermediate results on serverless.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One write (the most expensive step, because it computes and stores the result), then three actions that read the stored table instead of re-running the join. On Databricks, repeated reads also benefit from the disk cache.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spark_basics")
table = "workspace.spark_basics.lesson02_10_customer_totals"

In [0]:
timed("write once to a Delta table", lambda: expensive.write.mode("overwrite").saveAsTable(table))
stored = spark.table(table)
timed("action 1: count (from table)", lambda: stored.count())
timed("action 2: segments > 10 (from table)", lambda: stored.filter("segment > 10").count())
timed("action 3: top customer (from table)", lambda: stored.orderBy(F.desc("total")).first())
spark.sql(f"DROP TABLE IF EXISTS {table}")

## 5. The Databricks disk cache (automatic)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains the disk cache, which Databricks uses automatically for Parquet and Delta reads.<br><br>
<b>Why it matters</b><br>Databricks keeps local copies of remote Parquet/Delta files on the workers' fast local SSDs, in an efficient format, the first time they're read. Later reads of the same files skip cloud storage. It needs no code and is safe: it detects file changes. That's different from the Spark cache:<br><br>
| | Spark cache (<code>cache()</code>) | Databricks disk cache |<br>|---|---|---|<br>| What | Any DataFrame result | Remote Parquet / Delta files |<br>| Where | Executor memory / disk | Local SSD of the workers |<br>| How | Explicit, lazy | Automatic on supported compute (including serverless) |<br>| Survives restarts | No | No (per node) but refills automatically |<br>| Stale data risk | Yes, if the source changes | No, it validates files |
</div>

## 6. When caching hurts

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the situations where caching makes things worse.<br><br>
<b>Why it matters</b><br>Caching isn't free: it costs memory that execution needs, takes time to build, and can go stale.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Don't cache when...</b>
<ul><li>The DataFrame is used <b>once</b>: you pay to store it and never benefit</li><li>It's <b>cheap to recompute</b> (a simple filter on a Delta table, already served by the disk cache)</li><li>It's <b>huge</b> compared with executor memory: it evicts other data and causes spill or recomputation</li><li>The source <b>changes</b> and you need fresh results: the cache keeps the old snapshot</li><li>You <b>forget to unpersist</b>: memory stays occupied for the rest of the session</li></ul>
</div>

## Under the hood

```
df.cache()          -> marks the logical plan as "to be cached" (InMemoryRelation), nothing stored yet
first action        -> tasks compute partitions AND store them as cached blocks (BlockManager)
later actions       -> plan is rewritten to read InMemoryTableScan instead of recomputing
executor lost       -> its cached partitions are recomputed from lineage when next needed
unpersist()         -> blocks removed from all executors
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `cache` and `persist` are lazy markers.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> the first action after `cache` fills it.

On classic compute, a cached DataFrame's plan contains `InMemoryRelation` / `InMemoryTableScan`, and the **Storage** tab of the Spark UI shows how much is cached and where.

In [0]:
if CAN_CACHE:
    c = expensive.cache(); c.count()
    c.filter("segment = 3").explain()
    c.unpersist()
else:
    print("not supported on this compute")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>cache()</code> didn't make anything faster</b><br>
It's lazy. The first action after <code>cache()</code> still computes everything, and only later actions benefit. Or the DataFrame was used only once.<br><br>
<b>⛔ Problem: caching fails on serverless</b><br>The caching APIs aren't supported there. Materialize to a Delta table, and rely on the disk cache.<br><br>
<b>⛔ Problem: out of memory after caching several DataFrames</b><br>Cached data competes with execution memory. Unpersist what you no longer need, or use <code>DISK_ONLY</code>.<br><br>
<b>⛔ Problem: results don't reflect new source data</b><br>A cached DataFrame keeps the old snapshot. Unpersist, then re-read.<br><br>
<b>⛔ Problem: caching a different DataFrame than the one you reuse</b><br><code>df.filter(...)</code> after <code>cache()</code> is a new plan built on the cached one, which is fine. But <code>df2 = df.withColumn(...); df.cache()</code> doesn't cache <code>df2</code>'s extra work.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>cache()</code> and <code>persist()</code>?</b><br>
<code>cache()</code> is <code>persist()</code> with the default storage level, which is <code>MEMORY_AND_DISK</code> for DataFrames. <code>persist()</code> lets you choose a level, such as <code>MEMORY_ONLY</code>, <code>DISK_ONLY</code> or a replicated variant.<br><br>

<b>🎤 2. Is <code>cache()</code> an action?</b><br>
No, it's lazy. It marks the DataFrame, and the next action computes and stores it.<br><br>

<b>🎤 3. When should you cache?</b><br>
When an expensive DataFrame is reused by several actions, for example in iterative algorithms or when many outputs come from one joined dataset, and it fits comfortably in cluster memory.<br><br>

<b>🎤 4. When should you not cache?</b><br>
When the data is used once, is cheap to recompute, is too large for memory, or must reflect changing sources. And always unpersist when finished.<br><br>

<b>🎤 5. What happens to cached data if an executor fails?</b><br>
Its cached partitions are lost and Spark recomputes them from the lineage the next time they're needed, unless a replicated storage level kept a copy elsewhere.<br><br>

<b>🎤 6. How do you handle reuse on Databricks serverless, where caching isn't supported?</b><br>
Write the intermediate result to a Delta table and read it back. Repeated reads benefit from the automatic disk cache, and the table can be shared across tasks and sessions.<br><br>

<b>🎤 7. What is the Databricks disk cache?</b><br>
An automatic cache that stores copies of remote Parquet and Delta files on the workers' local SSDs, so repeated reads avoid cloud storage. It validates files, so it doesn't serve stale data.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. What is the default storage level of <code>DataFrame.cache()</code> in Spark 3 and later?</b><br>
A. <code>MEMORY_ONLY</code><br>
B. <code>MEMORY_AND_DISK</code><br>
C. <code>DISK_ONLY</code><br>
D. <code>OFF_HEAP</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A notebook on serverless compute reuses an expensive joined DataFrame in ten later queries. <code>df.cache()</code> raises an error. What is the recommended approach?</b><br>
A. Collect the DataFrame to the driver<br>
B. Write the joined result to a Delta table once and query the table<br>
C. Convert it to a global temporary view<br>
D. Repeat the join in each query<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Global temp views aren't supported on serverless either, and they wouldn't store the data anyway.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Time two actions on <code>expensive</code> without caching, then (if supported) with caching. Write down the numbers</li><li>Persist with <code>MEMORY_ONLY</code>, check <code>storageLevel</code>, then unpersist</li><li>Implement a helper <code>reuse(df, name)</code> that caches the DataFrame when caching is supported, and otherwise writes it to a Delta table and returns the table</li><li>In one sentence: why is <code>cache()</code> before a single <code>write</code> a waste?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Without and with caching
timed("no cache, action 1", lambda: expensive.count())
timed("no cache, action 2", lambda: expensive.agg(F.sum("total")).first())
if CAN_CACHE:
    c = expensive.cache(); c.count()
    timed("cached, action 1", lambda: c.count())
    timed("cached, action 2", lambda: c.agg(F.sum("total")).first())
    c.unpersist()

# 2. MEMORY_ONLY
if CAN_CACHE:
    m = expensive.persist(StorageLevel.MEMORY_ONLY); m.count()
    print(m.storageLevel); m.unpersist()

# 3. Works on both classic and serverless compute
def reuse(df, name):
    if CAN_CACHE:
        cached = df.cache(); cached.count()
        return cached
    df.write.mode("overwrite").saveAsTable(name)
    return spark.table(name)

r = reuse(expensive, table)
print("rows:", r.count())
if not CAN_CACHE:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
else:
    r.unpersist()

# 4. The data is computed once for the write anyway, so storing it in the cache costs time and memory with no reuse.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>cache()</code> = <code>persist(MEMORY_AND_DISK)</code>. Both are lazy, and the next action fills the cache</li><li>Cache only expensive DataFrames that are reused, and <code>unpersist()</code> when done</li><li>Cached data lives in executors, and lost partitions are recomputed from lineage</li><li>SQL <code>CACHE TABLE</code> is eager by default</li><li>Serverless: no Spark caching. Materialize to Delta, and the disk cache speeds up repeated reads</li><li>Caching can hurt: single use, cheap recompute, huge data, stale results</li></ul>
</div>

| Task | Code |
|---|---|
| Cache | `df.cache(); df.count()` |
| Persist with a level | `df.persist(StorageLevel.DISK_ONLY)` |
| Inspect | `df.storageLevel`, `df.is_cached` |
| Free | `df.unpersist()` |
| SQL | `CACHE TABLE t`, `UNCACHE TABLE t` |
| Serverless reuse | `df.write.saveAsTable("t")`, then `spark.table("t")` |

## Git commit

```
git add 02_spark_internals/10_caching_persistence.ipynb
git commit -m "add 02_10 caching persistence notebook"
```